In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
H5_PATH     = "../data_TEMP/merged_audio_500.h5" # <--- UPDATE THIS PATH
META_PATH   = "../data_TEMP/merged_metadata_500.parquet" # <--- UPDATE THIS PATH
BATCH_SIZE = 1      # Reduced batch size for memory optimization. Try 1 first. <--- MODIFIED
SAMPLE_RATE = 16_000
MAX_NUM_TURNS = 5

In [ ]:
from ConvoStyleDataset import ConvoStyleDataset, collate_pad

dataset = ConvoStyleDataset(
    h5_path=H5_PATH,
    meta_path=META_PATH,
    meta_columns=["transcription"],
    sample_rate=SAMPLE_RATE,
    num_turns=MAX_NUM_TURNS # Changed from 5 to 1 to allow loading single-turn utterances for diagnosis
    # max_len_sec=10.0, # Uncomment and adjust this value (e.g., 10 seconds) for further memory optimization.
    # no max_len_sec -- collate_pad handles variable lengths
)

from torch.utils.data import DataLoader
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_pad,
    num_workers=0,
)

In [ ]:
from transformers import BertTokenizer, BertModel, WavLMModel, Wav2Vec2FeatureExtractor

bert_tokenizer  = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model      = BertModel.from_pretrained("bert-base-uncased").half().to(device)  # using half() to limit gpu memory for quick tests

# WavLM only needs a feature extractor, not a full processor with tokenizer
wavlm_processor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model     = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").half().to(device)

for model in (bert_model, wavlm_model):
    for param in model.parameters():
        param.requires_grad = False
    model.eval()

print("Encoders loaded and frozen")


In [ ]:
from UtteranceEncoder import ModalityEncoder, SelfAttentivePooling, DualModalityEmbedder


text_encoder  = ModalityEncoder(bert_model,  SelfAttentivePooling(768).half()).to(device)
audio_encoder = ModalityEncoder(wavlm_model, SelfAttentivePooling(768).half()).to(device)

embedder = DualModalityEmbedder(
    text_encoder, audio_encoder, bert_tokenizer, wavlm_processor
).to(device)

print("Embedder initialized.")

In [ ]:
# testing embedder

embedder.eval()

# Re-define batch here as it was not found in the kernel state
batch = next(iter(loader))

text_emb, audio_emb = embedder(
    batch["audio"].to(device).half(),
    batch["lengths"],
    batch["transcription"],
    text_only=batch["text_only"].to(device),
)

print("text_emb shape :", text_emb.shape)   # (B, T, 768)
print("audio_emb shape:", audio_emb.shape)  # (B, T, 768)


assert text_emb.shape  == (BATCH_SIZE, MAX_NUM_TURNS, 768)
assert audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, 768)
assert not torch.allclose(text_emb, audio_emb), "embeddings are suspiciously identical"

print("All embedder assertions passed")

In [ ]:
from IntraModalDialogueEncoder import ContextAwareTransformer, SpeakerAwareTransformer

# Define transformer parameters
d_model = 768 # Embedding dimension from BERT/WavLM
nhead = 8 # Number of attention heads
num_encoder_layers = 2 # Number of transformer encoder layers
dim_feedforward = 2048 # Dimension of the feedforward network model

# Instantiate the ContextAwareTransformer and move it to the device
context_transformer = ContextAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device).half()

# pass audio and text embeddings separately
context_aware_embeddings_audio = context_transformer(audio_emb)
context_aware_embeddings_text = context_transformer(text_emb)

print("Context-aware audio embeddings shape:", context_aware_embeddings_audio.shape)
print("Context-aware audio embeddings shape:", context_aware_embeddings_text.shape)
assert context_aware_embeddings_audio.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert context_aware_embeddings_text.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
print("Context-aware audio transformer output shape assertion passed.")

# Instantiate the SpeakerAwareTransformer and move it to the device
speaker_aware_transformer = SpeakerAwareTransformer(
    d_model=d_model,
    nhead=nhead,
    num_encoder_layers=num_encoder_layers,
    dim_feedforward=dim_feedforward
).to(device).half()

# Pass audio embeddings through the speaker-aware transformer
intra_speaker_aware_audio_emb, inter_speaker_aware_audio_emb = speaker_aware_transformer(
    audio_emb,
    speaker_ids_list=batch["speaker_id"]
)

# Pass text embeddings through the speaker-aware transformer
intra_speaker_aware_text_emb, inter_speaker_aware_text_emb = speaker_aware_transformer(
    text_emb,
    speaker_ids_list=batch["speaker_id"]
)

print("Intra-speaker aware audio embeddings shape:", intra_speaker_aware_audio_emb.shape)
print("Inter-speaker aware audio embeddings shape:", inter_speaker_aware_audio_emb.shape)
print("Intra-speaker aware text embeddings shape:", intra_speaker_aware_text_emb.shape)
print("Inter-speaker aware text embeddings shape:", inter_speaker_aware_text_emb.shape)

assert intra_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert inter_speaker_aware_audio_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert intra_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
assert inter_speaker_aware_text_emb.shape == (BATCH_SIZE, MAX_NUM_TURNS, d_model)
print("Speaker-aware transformer output shapes assertion passed.")